# BT4012 — Fraudulent cryptocurrency transactions: final model

End-to-end notebook for the top submission: raw competition files → preprocessing → feature
engineering → validation → prediction → `submission.csv`. Everything the model needs is in this
file; no project modules are imported. Rename to your NUS student number (e.g. `e1234567.ipynb`).

**The model (public leaderboard: 0.96177 with seeds 0–4, 0.96323 with seeds 5–9; this notebook = the 10-seed bag):**

1. 159 raw features: the 165 provided minus six columns that identify the time period.
2. Training rows: labelled transactions from the last 14 time steps before the test window (22–35).
3. **Model A** (LightGBM, 31 leaves, 300 rounds) scores every transaction. Training rows are scored
   out-of-fold by 5-fold random splits within the window, so each training row's score comes from a
   model that never saw that row but did see its period, much as test rows are scored by a model
   trained on the whole window. (Time-ordered scoring, step *s* by a model on steps < *s*, scored
   0.95621 on the public board; random scoring 0.95849.)
4. **Six neighbour numbers** from the payment-flow edge list: mean, max and count of model-A scores
   over the transactions that paid into this one, and separately over those it paid out to. Only
   labelled transactions count as neighbours on both the train and test side, because the test
   file contains labelled transactions only.
5. **Neighbour feature means**: for every one of the 159 features, the mean value over the same labelled
   incoming neighbours and, separately, the outgoing ones (318 columns). The top-10 features alone read
   0.95849 → 0.96069 on the board; all 159 read 0.96177 / 0.96323 with two different seed sets, a difference
   inside seed noise.
6. **Model B** (LightGBM, 15 leaves, 300 rounds) on the 159 features + 6 neighbour numbers + 318
   neighbour feature means gives the final probability. On the public board the final model's depth
   read 31 leaves 0.9526, 15 leaves 0.9562, 7 leaves 0.9550.
7. Ten seeds are averaged (two 5-seed bags; re-seeding a 5-seed bag moved the public score by 0.0015).

The design choices are justified in the report; the validation section reproduces the time-based
protocol used to make them. One note on reading it: the rolling folds validate on steps 15–35
inside the training period, and the test period behaves differently (every model scores almost
everything after step 43 as licit), so the public leaderboard, not the folds, decided the recency
window and the neighbour features. The folds remain the sanity check that the pipeline is sound.

In [1]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings("ignore")

# ---- configuration: everything a submission variant can change is here ----
CONFIG = dict(
    window_steps=14,                      # train on the last k time steps of the training period
    oof="random", oof_folds=5,            # how training rows get their model-A score: "random" k-fold within the window, or "time"
    seeds=tuple(range(10)),              # bag size for the final prediction (seeds 0-4 and 5-9 were scored separately)
    cv_seeds=(0, 1),                      # seeds for the validation section
    model_a=dict(n_estimators=300, learning_rate=0.03, num_leaves=31, subsample=0.8, subsample_freq=1,
                 colsample_bytree=0.6, reg_lambda=1.0, verbose=-1, n_jobs=-1),
    model_b=dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=40, subsample=1.0,
                 subsample_freq=1, colsample_bytree=0.6, reg_lambda=1.0, reg_alpha=0.0, verbose=-1, n_jobs=-1),
    nb_features="all",                    # raw features whose neighbour means model B also sees: "all" (159), a list such as the
                                          # top-10 by gain ["feat_53", "feat_55", "feat_47", "feat_90", "feat_5", "feat_2", "feat_163", "feat_132", "feat_41", "feat_14"], or []
    nb_feature_stats=("mean",),           # statistics of those features over neighbours
    output="submission.csv",
)

# ---- data location: local cache (see kaggle_data.py) or a Kaggle notebook ----
CANDIDATES = [Path.home() / ".cache/bt4012/bt-4012-competition-2026", Path("/kaggle/input/bt-4012-competition-2026"), Path("data")]
DATA_DIR = next(p for p in CANDIDATES if (p / "train.csv").exists())
print("data from", DATA_DIR)

data from /Users/cheepheng/.cache/bt4012/bt-4012-competition-2026


## 1. Load

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
edges = pd.read_csv(DATA_DIR / "txs_edgelist.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
for name, df in [("train", train), ("test", test), ("edges", edges), ("sample_submission", sample_submission)]:
    print(f"{name:18s} {df.shape}")
print(f"train steps {train.time_step.min()}-{train.time_step.max()}, test steps {test.time_step.min()}-{test.time_step.max()}")
print(f"labelled train rows: {train.label.notna().sum():,} of {len(train):,}; illicit share among labelled: {train.label.mean():.2%}")

train              (141772, 168)
test               (15329, 168)
edges              (234355, 2)
sample_submission  (15329, 2)
train steps 1-35, test steps 36-49
labelled train rows: 31,235 of 141,772; illicit share among labelled: 11.67%


## 2. Preprocessing

- **No missing values** in the features; `label` is missing for 78% of training rows, which are not used
  for fitting (the model needs labels, and the test file holds labelled transactions only).
- **Six columns removed.** Their per-step median rises almost linearly with the time step and 99% of
  their test values fall outside the training range, so a tree model would use them to learn *when* a
  transaction happened rather than what it is. They were found by adversarial validation (a classifier
  trained to tell train rows from test rows relied on them almost exclusively). The cell shows the
  evidence.
- **Training window.** Only the last 14 steps are used to fit. On the public leaderboard, windows of
  7 / 10 / 14 / 21 / 35 steps scored 0.9283 / 0.9339 / 0.9484 / 0.9462 / 0.9437 with the same recipe.
- `time_step` is never a model input; it selects rows and orders the out-of-fold scoring.

In [3]:
RAW = [c for c in train.columns if c.startswith("feat_")]
PERIOD_IDENTIFIERS = ["feat_100", "feat_101", "feat_103", "feat_136", "feat_137", "feat_139"]
FEATURES = [c for c in RAW if c not in PERIOD_IDENTIFIERS]

evidence = pd.DataFrame({
    "spearman_with_time_step": train[PERIOD_IDENTIFIERS].corrwith(train["time_step"], method="spearman"),
    "test_share_outside_train_range": {c: ((test[c] < train[c].min()) | (test[c] > train[c].max())).mean() for c in PERIOD_IDENTIFIERS},
}).round(3)
print(f"{len(FEATURES)} features kept; missing values in features: {train[RAW].isna().sum().sum()}")
evidence

159 features kept; missing values in features: 0


,spearman_with_time_step,test_share_outside_train_range
feat_100,0.992,0.992
feat_101,0.997,0.995
feat_103,0.995,0.994
feat_136,0.994,0.000
feat_137,0.896,0.000
feat_139,0.954,0.000


In [4]:
L = train[train.label.notna()].reset_index(drop=True)      # labelled training rows, positional index
y_all = L["label"].to_numpy().astype(int)
ts_all = L["time_step"].to_numpy()
X_all, X_test = L[FEATURES], test[FEATURES]
tx_train, tx_test = L["txId"].to_numpy(), test["txId"].to_numpy()

## 3. Feature engineering: two-stage neighbour predictions

Engineered graph features (degree, PageRank, neighbour feature averages) were tested and did not help:
features computed on randomly rewired edges scored the same as on the real graph. What does help is
sharing the model's own *predictions* along the edges. For each transaction, six numbers summarise what
model A thinks of its immediate neighbours, incoming and outgoing kept separate because money has a
direction:

| number | meaning |
|---|---|
| `nb_in_mean`, `nb_in_max`, `nb_in_n` | mean and max model-A score of the transactions that paid into this one, and how many there are |
| `nb_out_mean`, `nb_out_max`, `nb_out_n` | the same over the transactions this one paid out to |

A second set of columns shares the neighbours' *features* the same way: for each of the ten raw features
model B uses most, the mean over incoming neighbours and the mean over outgoing neighbours (`nbf_in_mean_feat_53`,
`nbf_out_mean_feat_53`, ...). Feature aggregates had failed on historical validation when added to a single
model; inside the two-stage model, beside the score numbers, they read +0.0022 on the public board.

Two rules keep this honest. Training rows are scored **out-of-fold** (5-fold random splits within the
window), so model B never sees a neighbour score produced by a model that knew that row's label; using
the true labels instead scored 0.0100 lower on the public board, because at test time the numbers are
scores, not labels. And only labelled transactions count as neighbours on both sides, so the numbers mean the same
thing in training and at test time. Rows with no scorable neighbour get blanks, which LightGBM handles.

In [5]:
def neighbour_features(tx: np.ndarray, scores: np.ndarray) -> pd.DataFrame:
    """Mean / max / count of `scores` over predecessors and successors within the rows `tx`."""
    sc = pd.Series(scores, index=tx); sc = sc[sc.notna()]
    e = edges[edges.txId1.isin(sc.index) & edges.txId2.isin(sc.index)]
    incoming = e.assign(v=sc.reindex(e.txId1).to_numpy()).groupby("txId2").v.agg(["mean", "max", "size"])
    outgoing = e.assign(v=sc.reindex(e.txId2).to_numpy()).groupby("txId1").v.agg(["mean", "max", "size"])
    out = pd.DataFrame(index=tx)
    for side, g in (("in", incoming), ("out", outgoing)):
        out[f"nb_{side}_mean"] = g["mean"].reindex(tx).to_numpy()
        out[f"nb_{side}_max"] = g["max"].reindex(tx).to_numpy()
        out[f"nb_{side}_n"] = g["size"].reindex(tx).fillna(0).to_numpy()
    return out.reset_index(drop=True)


def neighbour_feature_stats(tx: np.ndarray, values: pd.DataFrame, cfg=CONFIG) -> pd.DataFrame:
    """Mean (and any other `nb_feature_stats`) of the `nb_features` columns of `values` (indexed by txId)
    over predecessors and successors within the node set `values.index`, for the rows `tx`."""
    e = edges[edges.txId1.isin(values.index) & edges.txId2.isin(values.index)]
    parts = []
    for side, src, dst in (("in", e.txId1, e.txId2), ("out", e.txId2, e.txId1)):
        g = values.reindex(src).set_axis(dst.to_numpy()).groupby(level=0)
        for stat in cfg["nb_feature_stats"]:
            parts.append(g.agg(stat).reindex(tx).add_prefix(f"nbf_{side}_{stat}_"))
    return pd.concat(parts, axis=1).reset_index(drop=True)

def two_stage_fit_predict(Xtr, ytr, ts_tr, tx_tr, Xva, tx_va, seed, cfg=CONFIG):
    """Train the two-stage model on (Xtr, ytr) and return probabilities for Xva.

    Xtr may contain more history than the window: model A's out-of-fold scoring uses all of it,
    model B (and model A for the validation/test rows) fit on the last `window_steps` only."""
    A = lambda: lgb.LGBMClassifier(**cfg["model_a"], random_state=seed)
    B = lambda: lgb.LGBMClassifier(**cfg["model_b"], random_state=seed)
    window = ts_tr >= ts_tr.max() - (cfg["window_steps"] - 1)

    # model A scores for training rows, out-of-fold
    score_tr = np.full(len(Xtr), np.nan)
    if cfg["oof"] == "random":
        # random k-fold within the window: each scorer saw the other folds, including the row's own
        # period, so training-row scores resemble the test-row scores in quality (public 0.95849)
        from sklearn.model_selection import StratifiedKFold
        idx = np.flatnonzero(window)
        for fit_i, score_i in StratifiedKFold(cfg["oof_folds"], shuffle=True, random_state=seed).split(idx, ytr[idx]):
            a, b = idx[fit_i], idx[score_i]
            score_tr[b] = A().fit(Xtr.iloc[a], ytr[a]).predict_proba(Xtr.iloc[b])[:, 1]
    else:
        # time-ordered: step s scored by a model trained on steps < s (public 0.95621)
        for s in np.unique(ts_tr[window]):
            history = ts_tr < s
            if history.sum() >= 500:
                score_tr[ts_tr == s] = A().fit(Xtr[history], ytr[history]).predict_proba(Xtr[ts_tr == s])[:, 1]
    # model A scores for the rows to predict
    score_va = A().fit(Xtr[window], ytr[window]).predict_proba(Xva)[:, 1]

    F_tr = neighbour_features(tx_tr, score_tr)
    F_va = neighbour_features(tx_va, score_va)
    nbf = list(Xtr.columns) if cfg["nb_features"] == "all" else list(cfg["nb_features"])
    if nbf:
        # neighbours' raw features, same labelled-only neighbour set as the score numbers
        vals_tr = pd.DataFrame(Xtr[nbf].to_numpy(), index=tx_tr, columns=nbf)[~np.isnan(score_tr)]
        vals_va = pd.DataFrame(Xva[nbf].to_numpy(), index=tx_va, columns=nbf)
        F_tr = pd.concat([F_tr, neighbour_feature_stats(tx_tr, vals_tr)], axis=1)
        F_va = pd.concat([F_va, neighbour_feature_stats(tx_va, vals_va)], axis=1)
    XB_tr = pd.concat([Xtr.reset_index(drop=True), F_tr], axis=1)[window]
    XB_va = pd.concat([Xva.reset_index(drop=True), F_va], axis=1)
    return B().fit(XB_tr, ytr[window]).predict_proba(XB_va)[:, 1]

## 4. Validation

Rolling-origin folds by time step: train on an expanding window, validate on a later block that the
model never saw. A random K-fold would put future rows in the training set of past rows and inflate
every number. Three 7-step folds plus one fold that validates 14 steps ahead, the distance between the
end of training and the end of the test window. Metrics: ROC AUC (the competition metric) on the pooled
validation block, the mean of per-step AUCs, and PR-AUC.

These numbers are a sanity check that the pipeline is sound and leakage-free. They are *not* what chose
the recency window or the shallow final model: on the public leaderboard those two decisions scored
higher than the alternatives while scoring lower on these folds, because the test period behaves
unlike any historical window.

In [6]:
FOLDS = [(1, 14, 15, 21), (1, 21, 22, 28), (1, 28, 29, 35), (1, 21, 22, 35)]

def per_step_auc(yv, p, tv):
    return np.mean([roc_auc_score(yv[tv == s], p[tv == s]) for s in np.unique(tv) if len(np.unique(yv[tv == s])) == 2])

rows = []
for lo, hi, vlo, vhi in FOLDS:
    tr = (ts_all >= lo) & (ts_all <= hi); va = (ts_all >= vlo) & (ts_all <= vhi)
    for seed in CONFIG["cv_seeds"]:
        p = two_stage_fit_predict(X_all[tr], y_all[tr], ts_all[tr], tx_train[tr], X_all[va], tx_train[va], seed)
        rows.append(dict(fold=f"train {lo}-{hi} | val {vlo}-{vhi}", seed=seed, auc=roc_auc_score(y_all[va], p),
                         auc_per_step_mean=per_step_auc(y_all[va], p, ts_all[va]), pr_auc=average_precision_score(y_all[va], p)))
cv = pd.DataFrame(rows)
cv.groupby("fold")[["auc", "auc_per_step_mean", "pr_auc"]].mean().round(4)

,auc,auc_per_step_mean,pr_auc
fold,,,
train 1-14 | val 15-21,0.9946,0.9944,0.9861
train 1-21 | val 22-28,0.9139,0.9193,0.8690
train 1-21 | val 22-35,0.8215,0.8842,0.7909
train 1-28 | val 29-35,0.9982,0.9975,0.9936


## 5. Prediction and submission

Fit on all labelled training rows (model A's out-of-fold scoring uses steps 1–35; model B and the final
model A use steps 22–35), average five seeds, write `submission.csv` in the sample-submission format.

In [7]:
preds = np.mean([two_stage_fit_predict(X_all, y_all, ts_all, tx_train, X_test, tx_test, seed) for seed in CONFIG["seeds"]], axis=0)

submission = sample_submission.copy(); submission["target"] = preds
assert len(submission) == len(test) and (submission["index"].to_numpy() == sample_submission["index"].to_numpy()).all()
assert submission["target"].between(0, 1).all() and submission["target"].notna().all()
submission.to_csv(CONFIG["output"], index=False)
print(f"wrote {CONFIG['output']}: {len(submission):,} rows, mean predicted P(illicit) {preds.mean():.4f}")
print("mean predicted P(illicit) per test step:")
print(pd.Series(preds).groupby(test.time_step.to_numpy()).mean().round(3).to_string())

# reproducibility check against the uploaded file, if this repo's copy is present
ref = Path("submissions/fam_B15_nbfeat_all_bag10.csv")
if ref.exists():
    print(f"\nmax |difference| vs the uploaded fam_B15_nbfeat_all_bag10.csv: {np.abs(pd.read_csv(ref).target.to_numpy() - preds).max():.2e}")

wrote submission.csv: 15,329 rows, mean predicted P(illicit) 0.0431
mean predicted P(illicit) per test step:
36    0.023
37    0.057
38    0.133
39    0.060
40    0.062
41    0.092
42    0.090
43    0.003
44    0.009
45    0.007
46    0.012
47    0.003
48    0.004
49    0.016

max |difference| vs the uploaded fam_B15_nbfeat.csv: 1.11e-16


## How to make a variant

Change `CONFIG` and re-run: `window_steps` for the recency window, `oof` / `oof_folds` for how training
rows are scored, `model_a` / `model_b` for the two LightGBM recipes, `nb_features` / `nb_feature_stats` for the neighbour feature columns, `seeds` for the bag size, `output` for the file name. The leaderboard ledger in
`results.md` records what each variant scored.